In [ ]:
import torch

print("Pytorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", device)

In [ ]:
!nvidia-smi

In [ ]:
%pip install -q \
    transformers \
    datasets \
    jiwer \
    librosa \
    soundfile \
    huggingface_hub \
    onnxruntime

print("Install complete")
print("No restart needed")

In [ ]:
import importlib.metadata as md

import transformers
import datasets
import jiwer
import librosa
import soundfile
import torchaudio
import onnxruntime

print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Torchaudio:", torchaudio.__version__)
print("Onnxruntime:", onnxruntime.__version__)
print("Huggingface hub:", md.version("huggingface_hub"))

print("IndicConformer environment setup complete")

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "ai4bharat/IndicVoices",
    "telugu",
    split="valid",
    streaming=True
)

num_samples = dataset.info.splits["valid"].num_examples

print(dataset)
print("IndicVoices Telugu validation samples:", num_samples)

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

CANDIDATES = [
    "ai4bharat/indic-conformer-600m-multilingual",
    "ai4bharat/indicconformer_stt_te_hybrid_rnnt_large",
    "ai4bharat/indicconformer_stt_te_hybrid_ctc_large",
]

available = []

for repo in CANDIDATES:
    try:
        info = api.model_info(repo)
        files = [s.rfilename for s in info.siblings]
        available.append(repo)
        print("OK      :", repo)
        print("          ", [f for f in files if f.endswith((".nemo", ".bin", ".safetensors"))][:5])
    except Exception as e:
        label = "GATED" if "Gated" in type(e).__name__ else "MISSING"
        print(f"{label:8}:", repo, "|", type(e).__name__)

if not available:
    raise RuntimeError(
        "No accessible IndicConformer repo. "
        "Request access at https://huggingface.co/ai4bharat/indic-conformer-600m-multilingual"
    )

MODEL_ID = available[0]
LANG_CODE = "te"
DECODING = "ctc"

print()
print("Model:", MODEL_ID)
print("Language:", LANG_CODE)
print("Decoding:", DECODING)

In [ ]:
import importlib

MODULES = [
    "transformers",
    "datasets",
    "huggingface_hub",
    "onnxruntime",
    "torchaudio",
    "soundfile",
    "librosa",
    "jiwer",
]

for mod in MODULES:
    try:
        importlib.import_module(mod)
        print("OK      :", mod)
    except Exception as e:
        print("FAIL    :", mod, "|", type(e).__name__, "|", str(e)[:120])

In [ ]:
import torch

from huggingface_hub import hf_hub_download, list_repo_files

device = "cuda" if torch.cuda.is_available() else "cpu"

repo_files = list_repo_files(MODEL_ID)
nemo_files = [f for f in repo_files if f.endswith(".nemo")]

nemo_available = False

if nemo_files:
    try:
        import nemo.collections.asr as nemo_asr
        nemo_available = True
    except Exception as e:
        print("NeMo not installed, falling back to transformers:", type(e).__name__)

if nemo_files and nemo_available:
    local_path = hf_hub_download(MODEL_ID, nemo_files[0])

    model = nemo_asr.models.ASRModel.restore_from(
        local_path,
        map_location=device
    )

    if hasattr(model, "change_decoding_strategy"):
        try:
            model.change_decoding_strategy(decoder_type=DECODING)
        except Exception as e:
            print("Decoding strategy unchanged:", e)

    BACKEND = "nemo"
else:
    from transformers import AutoModel

    try:
        model = AutoModel.from_pretrained(
            MODEL_ID,
            trust_remote_code=True
        )
    except ModuleNotFoundError as e:
        print("Remote code requires a module that is not installed:", e.name)
        raise

    BACKEND = "hf"

model = model.to(device)
model.eval()

print("Backend:", BACKEND)
print("Model loaded on:", device)

In [ ]:
import torch.nn as nn

def describe(module):
    params = list(module.parameters())
    total = sum(p.numel() for p in params)
    devices = {str(p.device) for p in params}
    dtypes = {str(p.dtype) for p in params}
    return total, devices, dtypes

print("Type:", type(model))
print("Is nn.Module:", isinstance(model, nn.Module))

total, devices, dtypes = describe(model)

print(f"Top level parameters: {total / 1e6:.1f} million")
print("Devices:", devices or "none")
print("Dtypes:", dtypes or "none")

print()
print("Named children:", [n for n, _ in model.named_children()])

submodules = {}

for name in dir(model):
    if name.startswith("_"):
        continue
    try:
        attr = getattr(model, name)
    except Exception:
        continue
    if isinstance(attr, nn.Module):
        count, devs, _ = describe(attr)
        if count:
            submodules[name] = (count, devs)

print()
print("Submodules holding parameters:")

for name, (count, devs) in sorted(submodules.items(), key=lambda kv: -kv[1][0]):
    print(f"  {name}: {count / 1e6:.1f}M on {devs}")

if not submodules and total == 0:
    print("  none loaded yet")

if torch.cuda.is_available():
    print()
    print(f"GPU memory allocated: {torch.cuda.memory_allocated() / (1024 ** 3):.2f} GB")

In [ ]:
import torchaudio

TARGET_SR = 16000

def load_waveform(sample_item):
    audio = sample_item["audio_filepath"].get_all_samples()

    waveform = audio.data
    sample_rate = audio.sample_rate

    if waveform.ndim == 1:
        waveform = waveform.unsqueeze(0)

    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    if sample_rate != TARGET_SR:
        waveform = torchaudio.functional.resample(
            waveform,
            orig_freq=sample_rate,
            new_freq=TARGET_SR
        )

    return waveform.float()

In [ ]:
import tempfile
import os
import soundfile as sf

def transcribe_waveform(waveform):
    if BACKEND == "hf":
        with torch.inference_mode():
            output = model(waveform.to(device), LANG_CODE, DECODING)

        text = output[0] if isinstance(output, (list, tuple)) else output

        return str(text).strip()

    tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
    tmp.close()

    try:
        sf.write(tmp.name, waveform.squeeze().cpu().numpy(), TARGET_SR)

        with torch.inference_mode():
            output = model.transcribe([tmp.name], batch_size=1, verbose=False)

        text = output[0] if isinstance(output, (list, tuple)) else output
        text = getattr(text, "text", text)

        return str(text).strip()
    finally:
        os.unlink(tmp.name)

def transcribe_sample(sample_item):
    return transcribe_waveform(load_waveform(sample_item))

In [ ]:
sample = next(iter(dataset))

waveform = load_waveform(sample)

print("Waveform shape:", waveform.shape)
print("Sample rate:", TARGET_SR)
print("Duration seconds:", waveform.shape[-1] / TARGET_SR)

In [ ]:
from IPython.display import Audio, display

display(
    Audio(
        waveform.squeeze().cpu().numpy(),
        rate=TARGET_SR
    )
)

In [ ]:
import time

start = time.time()

prediction = transcribe_waveform(waveform)

elapsed = time.time() - start

reference = sample["normalized"]

print("Reference :", reference)
print("Prediction:", prediction)
print()
print(f"Inference time: {elapsed:.2f} s")
print(f"Audio duration: {waveform.shape[-1] / TARGET_SR:.2f} s")

if torch.cuda.is_available():
    print(f"GPU memory allocated: {torch.cuda.memory_allocated() / (1024 ** 3):.2f} GB")

In [ ]:
from jiwer import process_words

def normalize_for_wer(text):
    return " ".join(str(text).strip().split())

if "reference" in globals() and "prediction" in globals():
    reference_clean = normalize_for_wer(reference)
    prediction_clean = normalize_for_wer(prediction)

    result = process_words(
        reference_clean,
        prediction_clean
    )

    print("WER:", result.wer)
    print("WER %:", result.wer * 100)
    print("Substitutions:", result.substitutions)
    print("Deletions:", result.deletions)
    print("Insertions:", result.insertions)
    print("Correct words:", result.hits)
else:
    print("normalize_for_wer ready")

In [ ]:
from itertools import islice

test_samples = list(islice(dataset, 20))

print("Samples collected:", len(test_samples))

In [ ]:
results = []

for i, sample_item in enumerate(test_samples):
    reference = normalize_for_wer(sample_item["normalized"])

    prediction = normalize_for_wer(transcribe_sample(sample_item))

    score = process_words(
        reference,
        prediction
    )

    results.append({
        "index": i,
        "speaker_id": sample_item["speaker_id"],
        "duration": sample_item["duration"],
        "reference": reference,
        "prediction": prediction,
        "wer": score.wer,
        "substitutions": score.substitutions,
        "deletions": score.deletions,
        "insertions": score.insertions
    })

    print(f"{i + 1}/20 complete")

In [ ]:
import pandas as pd

results_df = pd.DataFrame(results)

for i in range(5):
    print("REFERENCE :", results_df.iloc[i]["reference"])
    print("PREDICTION:", results_df.iloc[i]["prediction"])
    print("WER       :", results_df.iloc[i]["wer"])
    print()

results_df.head()

In [ ]:
smoke_result = process_words(
    results_df["reference"].tolist(),
    results_df["prediction"].tolist()
)

print("Smoke test samples:", len(results_df))
print("Corpus WER:", smoke_result.wer)
print("Corpus WER %:", smoke_result.wer * 100)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path

results_dir = Path(
    "/content/drive/MyDrive/DhwaniLab/results/indicconformer"
)

results_dir.mkdir(
    parents=True,
    exist_ok=True
)

checkpoint_file = results_dir / "indicvoices_telugu_valid.csv"

print("Results directory:", results_dir)
print("Checkpoint file:", checkpoint_file)
print("Checkpoint exists:", checkpoint_file.exists())

In [ ]:
import pandas as pd

if checkpoint_file.exists():
    saved_df = pd.read_csv(checkpoint_file)
    completed_indices = set(saved_df["index"].astype(int).tolist())
    all_results = saved_df.to_dict("records")
else:
    completed_indices = set()
    all_results = []

contiguous = completed_indices == set(range(len(completed_indices)))

print("Already completed:", len(completed_indices))
print("Remaining:", num_samples - len(completed_indices))

if completed_indices:
    print("Highest index saved:", max(completed_indices))
    print("Contiguous from zero:", contiguous)

In [ ]:
save_every = 25
new_since_save = 0

if completed_indices and contiguous:
    start_index = len(completed_indices)
    stream = dataset.skip(start_index)
else:
    start_index = 0
    stream = dataset

print("Resuming at index:", start_index)

for offset, sample_item in enumerate(stream):

    index = start_index + offset

    if index in completed_indices:
        continue

    reference = normalize_for_wer(
        sample_item["normalized"]
    )

    prediction = normalize_for_wer(
        transcribe_sample(sample_item)
    )

    score = process_words(
        reference,
        prediction
    )

    row = {
        "index": index,
        "speaker_id": sample_item["speaker_id"],
        "duration": sample_item["duration"],
        "reference": reference,
        "prediction": prediction,
        "wer": score.wer,
        "substitutions": score.substitutions,
        "deletions": score.deletions,
        "insertions": score.insertions,
    }

    all_results.append(row)
    completed_indices.add(index)

    new_since_save += 1

    print(
        f"{index + 1}/{num_samples} | "
        f"WER: {score.wer:.3f}"
    )

    if new_since_save >= save_every:

        checkpoint_df = pd.DataFrame(
            all_results
        ).sort_values("index")

        checkpoint_df.to_csv(
            checkpoint_file,
            index=False
        )

        print(
            f"Checkpoint saved: "
            f"{len(checkpoint_df)} samples"
        )

        new_since_save = 0

final_checkpoint_df = pd.DataFrame(all_results).sort_values("index")

final_checkpoint_df.to_csv(checkpoint_file, index=False)

print("Final save:", len(final_checkpoint_df), "samples")

In [ ]:
final_df = pd.read_csv(checkpoint_file)

references = final_df["reference"].fillna("").tolist()
predictions = final_df["prediction"].fillna("").tolist()

final_result = process_words(
    references,
    predictions
)

reference_words = (
    final_result.hits
    + final_result.substitutions
    + final_result.deletions
)

print("Samples:", len(final_df))
print("Reference words:", reference_words)
print("Correct words:", final_result.hits)
print("Substitutions:", final_result.substitutions)
print("Deletions:", final_result.deletions)
print("Insertions:", final_result.insertions)
print("Corpus WER:", final_result.wer)
print("Corpus WER %:", final_result.wer * 100)

In [ ]:
summary = {
    "model": MODEL_ID,
    "decoding": DECODING,
    "dataset": "ai4bharat/IndicVoices telugu valid",
    "samples": len(final_df),
    "reference_words": int(reference_words),
    "hits": int(final_result.hits),
    "substitutions": int(final_result.substitutions),
    "deletions": int(final_result.deletions),
    "insertions": int(final_result.insertions),
    "corpus_wer": float(final_result.wer),
    "corpus_wer_percent": float(final_result.wer * 100),
}

summary_file = results_dir / "indicconformer_summary.csv"

pd.DataFrame([summary]).to_csv(summary_file, index=False)

print("Summary saved to:", summary_file)
print(summary)